# OCR Scanner

This notebook demonstrates how to perform Optical Character Recognition (OCR) on images using Tesseract.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import pytesseract
import numpy as np
import os

# Configure pytesseract path if needed (uncomment and modify if Tesseract is not in PATH)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Function to configure Tesseract
def configure_tesseract():
    """Configure pytesseract to work with Tesseract OCR."""
    # Common Tesseract installation paths on Windows
    possible_paths = [
        r'C:\Program Files\Tesseract-OCR\tesseract.exe',
        r'C:\Program Files (x86)\Tesseract-OCR\tesseract.exe',
        r'C:\Users\anupa\AppData\Local\Programs\Tesseract-OCR\tesseract.exe',
        r'D:\Tesseract-OCR\tesseract.exe'
    ]
    
    # Try to find Tesseract in common locations
    for path in possible_paths:
        if os.path.exists(path):
            pytesseract.pytesseract.tesseract_cmd = path
            print(f"Tesseract found at: {path}")
            return True
    
    # If not found, try to use it from PATH
    try:
        version = pytesseract.get_tesseract_version()
        print(f"Tesseract found in PATH. Version: {version}")
        return True
    except pytesseract.TesseractNotFoundError:
        print("Tesseract not found. Please install Tesseract OCR.")
        return False

# Configure Tesseract
TESSERACT_CONFIGURED = configure_tesseract()

In [ ]:
# Load an image with text
image = cv2.imread("../data/test_images/car.jpg")  # This might not have text, but for demo
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.imshow(image)
plt.title("Input Image")
plt.show()

# Preprocessing for better OCR results
def preprocess_for_ocr(image):
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Apply threshold to get binary image
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return gray, thresh

gray, processed = preprocess_for_ocr(image)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(gray, cmap='gray')
ax1.set_title("Grayscale")
ax2.imshow(processed, cmap='gray')
ax2.set_title("Processed for OCR")
plt.show()

In [ ]:
# Perform OCR
def perform_ocr(image):
    if not TESSERACT_CONFIGURED:
        raise Exception(
            "Tesseract OCR is not installed or not found in PATH.\n\n"
            "To fix this:\n"
            "1. Download Tesseract from: https://github.com/UB-Mannheim/tesseract/wiki\n"
            "2. Install it (choose the option to add to PATH during installation)\n"
            "3. Or manually set the path in the code if installed in a custom location\n\n"
            "Common installation paths:\n"
            "- C:\\Program Files\\Tesseract-OCR\\tesseract.exe\n"
            "- C:\\Program Files (x86)\\Tesseract-OCR\\tesseract.exe"
        )

    # Preprocess the image
    gray, processed = preprocess_for_ocr(image)

    # Extract text using pytesseract
    text = pytesseract.image_to_string(processed)

    # Get detailed data including bounding boxes
    data = pytesseract.image_to_data(processed, output_type=pytesseract.Output.DICT)

    return text, data, processed

# Test OCR
if TESSERACT_CONFIGURED:
    extracted_text, ocr_data, processed_image = perform_ocr(image)

    print("Extracted Text:")
    print(extracted_text)

    # Visualize detected text regions
    def draw_text_boxes(image, data):
        img_copy = image.copy()
        n_boxes = len(data['text'])

        for i in range(n_boxes):
            if int(data['conf'][i]) > 60:  # Only show confident detections
                (x, y, w, h) = (data['left'][i], data['top'][i], data['width'][i], data['height'][i])
                cv2.rectangle(img_copy, (x, y), (x + w, y + h), (0, 255, 0), 2)

        return img_copy

    result_image = draw_text_boxes(image, ocr_data)

    plt.imshow(result_image)
    plt.title("Detected Text Regions")
    plt.show()
else:
    print("Tesseract not configured. Please install Tesseract OCR first.")